In [1]:
import numpy as np
import pandas as pd
import cooler
from cooltools.lib.numutils import observed_over_expected, adaptive_coarsegrain
from cooltools.lib.numutils import set_diag, interp_nan
from astropy.convolution import Gaussian2DKernel
from astropy.convolution import convolve

In [2]:
import sys
import os
sys.path.append(os.path.abspath("/home1/smaruj/pytorch_akita/"))

# from model import SeqNN
from model_v2_compatible import SeqNN

In [3]:
import torch
from pyfaidx import Fasta

In [4]:
flame_df_path = "/scratch1/smaruj/stripepy_stripes/selected_stripes.tsv"

In [5]:
flame_df = pd.read_csv(flame_df_path, sep="\t")

In [6]:
flame_df 

,chrom1,start1,end1,chrom2,start2,end2,rel_change,frac_missing_bbox_pm5bins_bins,x_length,y_length,orientation,midpoint,window_start,window_end
0,chr1,3121152,4136960,chr1,3096576,3153920,20.008292,0.0,1015808,57344,horizontal,3121152,2465792,3776512
1,chr1,3178496,4341760,chr1,3162112,3194880,31.519521,0.0,1163264,32768,horizontal,3178496,2523136,3833856
2,chr1,3244032,3694592,chr1,3244032,3260416,29.272392,0.0,450560,16384,horizontal,3244032,2588672,3899392
3,chr1,3801088,3850240,chr1,3170304,3833856,17.663422,0.0,49152,663552,vertical,3833856,3178496,4489216
4,chr1,4005888,4055040,chr1,3358720,4030464,24.861986,0.0,49152,671744,vertical,4030464,3375104,4685824
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6836,chrY,2514944,2547712,chrY,2105344,2514944,26.819652,0.0,32768,409600,vertical,2514944,1859584,3170304
6837,chrY,4243456,4341760,chrY,4243456,4259840,17.552819,0.0,98304,16384,horizontal,4243456,3588096,4898816
6838,chrY,4276224,4358144,chrY,4259840,4308992,20.848509,0.0,81920,49152,horizontal,4276224,3620864,4931584
6839,chrY,4317184,4349952,chrY,4268032,4333568,15.210400,0.0,32768,65536,vertical,4333568,3678208,4988928


In [7]:
FASTA_FILE = "/project/fudenber_735/genomes/mm10/mm10.fa"
COOL_FILE = "/project/fudenber_735/GEO/Hsieh2019/4DN/mESC_mm10_4DNFILZ1CPT8.mapq_30.2048.cool"

# --- Load Data ---
genome = Fasta(FASTA_FILE)

genome_hic_cool = cooler.Cooler(COOL_FILE)

In [8]:
from cooltools import coverage

In [17]:
cov = coverage(genome_hic_cool, nproc=50, clr_weight_name='weight')

INFO:root:creating a Pool of 50 workers


In [18]:
cov

array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]])

In [21]:
np.nanmin(cov), np.nanmean(cov)

(0.0, 0.9281126788713165)

In [ ]:
import random

def one_hot_encode_sequence(sequence_obj):
    sequence = str(sequence_obj).upper()
    base_to_int = {'A': 0, 'C': 1, 'G': 2, 'T': 3}

    encoded_sequence = np.array([
        base_to_int.get(base, base_to_int[random.choice("ACGT")]) for base in sequence
    ])

    one_hot_encoded = np.zeros((4, len(encoded_sequence)), dtype=np.float32)
    one_hot_encoded[encoded_sequence, np.arange(len(encoded_sequence))] = 1

    return np.expand_dims(one_hot_encoded, axis=0)

In [ ]:
def process_hic_matrix(genome_hic_cool, mseq_str, diagonal_offset=2, padding=64, kernel_stddev=1):
    seq_hic_raw = genome_hic_cool.matrix(balance=True).fetch(mseq_str)
    
    # Check for NaN filtering percentage
    seq_hic_nan = np.isnan(seq_hic_raw)
    num_filtered_bins = np.sum(np.sum(seq_hic_nan, axis=0) == len(seq_hic_nan))
    print("num_filtered_bins:", num_filtered_bins)
    
    if num_filtered_bins > (0.5 * len(seq_hic_nan)):
        print(f"More than 50% bins filtered in {mseq_str}. Check Hi-C data quality.")
        
    # clip first diagonals and high values
    clipval = np.nanmedian(np.diag(seq_hic_raw, diagonal_offset))
    for i in range(-diagonal_offset+1, diagonal_offset):
        set_diag(seq_hic_raw, clipval, i)
    seq_hic_raw = np.clip(seq_hic_raw, 0, clipval)
    seq_hic_raw[seq_hic_nan] = np.nan
    
    # adaptively coarsegrain based on raw counts
    seq_hic_smoothed = adaptive_coarsegrain(
                            seq_hic_raw,
                            genome_hic_cool.matrix(balance=False).fetch(mseq_str),
                            cutoff=2, max_levels=8)
    seq_hic_nan = np.isnan(seq_hic_smoothed)
    
    # local obs/exp
    seq_hic_obsexp = observed_over_expected(seq_hic_smoothed, ~seq_hic_nan)[0]
    
    log_hic_obsexp = np.log(seq_hic_obsexp)
    
    # Apply padding
    if padding > 0:
        log_hic_obsexp = log_hic_obsexp[padding:-padding, padding:-padding]
    
    log_hic_obsexp = interp_nan(log_hic_obsexp)
    for i in range(-diagonal_offset+1, diagonal_offset): set_diag(log_hic_obsexp, 0,i)
    
    kernel = Gaussian2DKernel(x_stddev=kernel_stddev)
    seq_hic = convolve(log_hic_obsexp, kernel)
    
    return seq_hic    

In [ ]:
def upper_triangular_to_vector_skip_diagonals(matrix, dim=512, diag=2):
    
    # Extract the upper triangular part excluding the first two diagonals
    upper_triangular_vector = matrix[np.triu_indices(dim, k=diag)]
    
    return upper_triangular_vector

In [ ]:
N = 256
diagonal_offset = 2

In [ ]:
# --- Load model ---
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

model = SeqNN()
model.load_state_dict(torch.load("/home1/smaruj/pytorch_akita/model_0_v2_finetuned_correctly.pt", map_location=device))
model.eval()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
def plot_map(matrix, vmin=-0.6, vmax=0.6, palette="RdBu_r", width=5, height=5):
    fig, axes = plt.subplots(1, 1, figsize=(width, height))

    sns.heatmap(
        matrix,
        vmin=vmin,
        vmax=vmax,
        cbar=False,
        cmap=palette,
        square=True,
        xticklabels=False,
        yticklabels=False,
        ax=axes
    )

    plt.tight_layout()
    plt.show()

In [ ]:
flame_df.columns

In [ ]:
# Helper function to set diagonal elements to a specific value
def set_diag(matrix, value, k):
    # Explicitly set the diagonal to 'value' (in this case, np.nan) for each k
    rows, cols = matrix.shape
    for i in range(rows):
        if 0 <= i + k < cols:
            matrix[i, i + k] = value

def from_upper_triu(vector_repr, matrix_len, num_diags):
    # Ensure vector_repr is a NumPy array (if it's a PyTorch tensor, convert it)
    if isinstance(vector_repr, torch.Tensor):
        vector_repr = vector_repr.detach().flatten().cpu().numpy()  # Flatten and convert to NumPy array

    # Initialize a zero matrix of shape (matrix_len, matrix_len)
    z = np.zeros((matrix_len, matrix_len))

    # Get the indices for the upper triangular matrix
    triu_tup = np.triu_indices(matrix_len, num_diags)

    # Assign the values from the vector_repr to the upper triangular part of the matrix
    z[triu_tup] = vector_repr

    # Set the diagonals specified by num_diags to np.nan
    for i in range(-num_diags + 1, num_diags):
        set_diag(z, np.nan, i)

    # Ensure the matrix is symmetric
    return z + z.T

In [ ]:
from scipy.stats import pearsonr

In [ ]:
flame_df

In [ ]:
pearson_r = []

map_midbin = 256

for i, row in enumerate(flame_df[:2].itertuples(index=False)):
    print("index:", i)
    chrom, start, end = row.chrom1, row.window_start, row.window_end
    mseq_str = f"{chrom}:{start}-{end}"
    
    # TARGET
    sequence = genome[chrom][start:end]
    ohe_sequence = one_hot_encode_sequence(sequence)
    tensor_ohe_sequence = torch.from_numpy(ohe_sequence)
    matrix = process_hic_matrix(genome_hic_cool, mseq_str, diagonal_offset=2, padding=64, kernel_stddev=1)
    target_vector = upper_triangular_to_vector_skip_diagonals(matrix)
    
    # print("target")
    # plot_map(matrix)

    # x_len, y_len = row.x_length // 2048, row.y_length // 2048

    # x_end = map_midbin + x_len
    # y_start = map_midbin - y_len
    
    # if x_end > 512:
    #     x_end = 512
    # if y_start < 0:
    #     y_start = 0
    
    # flame_mean = np.nanmean(matrix[map_midbin:x_end, y_start:map_midbin])

    # print("flame mean", flame_mean)
    
    # fig, ax = plt.subplots(figsize=(5, 5))
    # sns.heatmap(matrix, cmap="RdBu_r", center=0, square=True, ax=ax, vmin=-0.6, vmax=0.6)

    # # Draw vertical and horizontal lines at dot_r and dot_c
    # ax.axhline(map_midbin, color='black', linestyle='--')
    # ax.axhline(y_start, color='black', linestyle='--')
    # ax.axvline(x_end, color='black', linestyle='--')
    # ax.axvline(map_midbin, color='black', linestyle='--')
    
    # ax.set_title(f"Target")
    # plt.show()
    
    akita_pred = model(tensor_ohe_sequence.to(device)).cpu()
    # akita_map = from_upper_triu(akita_pred, 512, 2)
    torch_vec_np = akita_pred.squeeze().detach().cpu().numpy()

    # print("prediction")
    # plot_map(akita_map)
    
    # pred_flame_mean = np.nanmean(akita_map[map_midbin:x_end, y_start:map_midbin])

    # print("pred flame mean", pred_flame_mean)
    
    # fig, ax = plt.subplots(figsize=(5, 5))
    # sns.heatmap(akita_map, cmap="RdBu_r", center=0, square=True, ax=ax, vmin=-0.6, vmax=0.6)

    # # Draw vertical and horizontal lines at dot_r and dot_c
    # ax.axhline(map_midbin, color='black', linestyle='--')
    # ax.axhline(y_start, color='black', linestyle='--')
    # ax.axvline(x_end, color='black', linestyle='--')
    # ax.axvline(map_midbin, color='black', linestyle='--')
    
    # ax.set_title(f"Pred")
    # plt.show()
    
    # Pearson R
    r, p_value = pearsonr(target_vector, torch_vec_np)
    # print(f"Pearson R: {r:.4f}, p-value: {p_value:.4e}")
    
    pearson_r.append(r)

In [ ]:
flame_df["PearsonR"] = pearson_r

In [ ]:
pearson_r

In [ ]:
# flame_df.to_csv(f"/scratch1/smaruj/stripepy_stripes/selected_stripes_pearsonr.tsv", sep="\t", index=False)